<a href="https://colab.research.google.com/github/Polqer/diplommel1/blob/main/5attempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install efficientnet-pytorch
!git clone https://github.com/Polqer/diplommel1.git
!pip install torchcam


Cloning into 'diplommel1'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 392 (delta 17), reused 14 (delta 4), pack-reused 351 (from 2)
Receiving objects: 100% (392/392), 4.52 GiB | 33.01 MiB/s, done.
Resolving deltas: 100% (36/36), done.
Updating files: 100% (297/297), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 

In [3]:
import time
import copy
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
import gc
import torch.nn as nn
import wandb
import torch.optim as optim
import torch.nn.functional as F
from transformers import ViTForImageClassification

# from efficientnet_pytorch import EfficientNet
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from scipy.io import loadmat
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from google.colab import drive
#from torchcam.methods import GradCAM
#from torchcam.utils import overlay_mask
from timm import create_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drive.mount('/content/drive')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


train_dir = '/content/diplommel1/datamat/train'
val_dir = '/content/diplommel1/datamat/test'

vit_path = '/content/drive/MyDrive/vit_b_16_best.pt'

wandb.init(project="my-awesome-project", name= "matEfficientnet")

batch_size = 16
num_epochs = 20
learning_rate = 0.0001

num_classes = 3  # "mm", "nn", "other"

transform = transforms.Compose([
    transforms.Resize((224, 224)),  #Приводим изображение к размеру 224x224
    #transforms.ToTensor(),  #Преобразуем изображение в тензор
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  #Нормализуем
])

#т.к. данные являются несбалансированными Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2., alpha=0.25, num_classes=3):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        inputs = torch.clamp(inputs, 1e-7, 1 - 1e-7)  #чтобы избежать логарифмирования 0
        targets = torch.eye(self.num_classes).to(inputs.device).index_select(dim=0, index=targets)  #Преобразуем метки в one-hot
        cross_entropy_loss = -targets * torch.log(inputs)
        loss = self.alpha * torch.pow(1 - inputs, self.gamma) * cross_entropy_loss
        return loss.sum(dim=1).mean()  #Среднее по батчу

#датасет
# Определение датасета
class HyperspectralDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.files = [(file_path, self.class_to_idx[cls_name])
                      for cls_name in self.classes
                      for file_path in (self.root_dir / cls_name).glob("*.mat")]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path, label = self.files[idx]
        mat_data = loadmat(file_path)
        if 'DataCubeC' not in mat_data:
            raise KeyError(f"Ключ 'DataCubeC' не найден в файле {file_path}")
        image = mat_data['DataCubeC']
        image = np.transpose(image, (2, 0, 1))  # Транспонируем перед применением Resize

        #Преобразование в RGB: усреднение каналов для имитации красного, зелёного и синего спектров
        red_channel = np.mean(image[0:5], axis=0)  # Имитация красного
        green_channel = np.mean(image[5:10], axis=0)  # Имитация зелёного
        blue_channel = np.mean(image[10:15], axis=0)  # Имитация синего
        image = np.stack([red_channel, green_channel, blue_channel], axis=0)

        #Преобразуем в тензор
        image = torch.tensor(image, dtype=torch.float32)

        if self.transform:
          image = self.transform(image)

        return image, label




class ViT(nn.Module):
    def __init__(self, num_classes):
        super(ViT, self).__init__()

        # Используем предобученную модель ViT
        self.model = create_model('vit_base_patch16_224', pretrained=True)  # Модель ViT с патчами 16x16

        # Если у вас гиперспектральные изображения с 3 усредненными каналами, измените первый слой
        in_channels = 3  # Количество каналов в гиперспектральных изображениях
        self.model.patch_embed.proj = nn.Conv2d(in_channels=in_channels,
                                                out_channels=self.model.patch_embed.proj.out_channels,
                                                kernel_size=self.model.patch_embed.proj.kernel_size,
                                                stride=self.model.patch_embed.proj.stride,
                                                padding=self.model.patch_embed.proj.padding,
                                                bias=False)

        #Заменим последний слой классификации на количество классов
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

#Расчет метрик
def calculate_metrics(y_true, y_pred, y_true_proba, y_pred_proba):
    return {
        'accuracy': accuracy_score(y_true=y_true, y_pred=y_pred),
        'confusion_matrix': confusion_matrix(y_true=y_true, y_pred=y_pred),
        'micro/precision': precision_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0),
        'micro/recall': recall_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0),
        'micro/f1': f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0),
        'micro/roc_auc_score': roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='macro'),
        'macro/precision': precision_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0),
        'macro/recall': recall_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0),
        'macro/f1': f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0),
        'roc_auc_score':  roc_auc_score(y_true=y_true_proba, y_score=y_pred_proba, average=None, multi_class='ovr'),
        'weighted/precision': precision_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0),
        'weighted/recall': recall_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0),
        'weighted/f1': f1_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0)
    }



#Загрузка данных
train_dataset = HyperspectralDataset(train_dir, transform=transform)
val_dataset = HyperspectralDataset(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

#Инициализация модели
# model = ViT(num_classes=num_classes).to(device)
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224", attn_implementation="sdpa", torch_dtype=torch.float16)
# Загрузка состояния модели из checkpoint
#checkpoint = torch.load(checkpoint_path, map_location=device)



# Загрузка всех параметров модели, если их структура совпадает
# model.load_state_dict(checkpoint, strict=False)  # strict=False позволяет игнорировать лишние параметры
#model.load_state_dict(torch.load(vit_path))
model.load_state_dict(torch.load(vit_path, map_location=torch.device('cpu')))

# Функция потерь и оптимизатор
criterion = FocalLoss(gamma=2.0, alpha=0.25, num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Функция обучения
def train_model(model, criterion, optimizer, num_epochs, checkpoint_dir="/content/diplommel1/checkpoint"):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
# Создаём папку для чекпоинтов, если её нет
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(checkpoint_dir, "best_model.ckpt")
    # История обучения
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}\n' + '-' * 10)

        for phase, loader in [('train', train_loader), ('val', val_loader)]:
            is_train = phase == 'train'
            model.train(is_train)

            running_loss = 0.0
            running_corrects = 0

            all_labels, all_preds, all_true_proba, all_pred_proba = [], [], [], []

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(is_train):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    probabilities = torch.softmax(outputs, dim=1).detach().cpu().numpy()

                    if is_train:
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_true_proba.extend(np.eye(outputs.shape[1])[labels.cpu().numpy()])
                all_pred_proba.extend(probabilities)

            # Средние значения за эпоху
            dataset_size = len(loader.dataset)
            epoch_loss = running_loss / dataset_size
            epoch_acc = (running_corrects.double() / dataset_size).item()

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                # Вычисление метрик
                try:
                    metrics = calculate_metrics(
                        y_true=all_labels,
                        y_pred=all_preds,
                        y_true_proba=np.array(all_true_proba),
                        y_pred_proba=np.array(all_pred_proba)
                    )
                    wandb.log({
                        "Val Loss": epoch_loss,
                        "Val Accuracy": epoch_acc,
                        **{f"Val {k}": v for k, v in metrics.items() if k != 'confusion_matrix'},
                        "Epoch": epoch
                    })
                except Exception as e:
                    print(f"Ошибка при расчете метрик: {e}")

                # Обновление лучшей модели
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    # Сохранение модели в формате .ckpt
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': epoch_loss,
                        'accuracy': epoch_acc
                    }, checkpoint_path)
                    print(f'Модель сохранена в {checkpoint_path}')

        torch.cuda.empty_cache()
        gc.collect()

    #Время обучения
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)

    #Построение графиков
    plt.figure(figsize=(12, 5))

    #График Loss
    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label='Train Loss')
    plt.plot(history["val_loss"], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss')

    #График Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label='Train Accuracy')
    plt.plot(history["val_acc"], label='Val Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy')



    return model
#Вызов функции обучения
train_model(model, criterion, optimizer, num_epochs)


plt.xlabel("Epochs")
plt.ylabel("F1-score")
plt.title("Train vs Val F1-score")
plt.legend()
plt.grid(True)
plt.show()
wandb.finish()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu


<ipython-input-3-b5cfadc3d8f3>:170: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(vit_path, map_location=torch.device('cpu')))


RuntimeError: Error(s) in loading state_dict for ViTForImageClassification:
	Missing key(s) in state_dict: "vit.embeddings.cls_token", "vit.embeddings.position_embeddings", "vit.embeddings.patch_embeddings.projection.weight", "vit.embeddings.patch_embeddings.projection.bias", "vit.encoder.layer.0.attention.attention.query.weight", "vit.encoder.layer.0.attention.attention.query.bias", "vit.encoder.layer.0.attention.attention.key.weight", "vit.encoder.layer.0.attention.attention.key.bias", "vit.encoder.layer.0.attention.attention.value.weight", "vit.encoder.layer.0.attention.attention.value.bias", "vit.encoder.layer.0.attention.output.dense.weight", "vit.encoder.layer.0.attention.output.dense.bias", "vit.encoder.layer.0.intermediate.dense.weight", "vit.encoder.layer.0.intermediate.dense.bias", "vit.encoder.layer.0.output.dense.weight", "vit.encoder.layer.0.output.dense.bias", "vit.encoder.layer.0.layernorm_before.weight", "vit.encoder.layer.0.layernorm_before.bias", "vit.encoder.layer.0.

In [ ]:
import os

checkpoint_path = "/content/efficientnet_b0_best.pt"

if os.path.exists(checkpoint_path):
    print("Файл найден!")
else:
    print("Файл НЕ найден!")


Файл НЕ найден!


Попытка использовать Gradcam

Не получается реализовать, либо выдает, что 16 каналов надо 3, при изменениях пишет, что надо каналов 3 получил 16.



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask
from scipy.io import loadmat
import torch.nn as nn
from efficientnet_pytorch import EfficientNet
from PIL import Image
class EfficientNet1C(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNet1C, self).__init__()

        self.model = EfficientNet.from_pretrained('efficientnet-b0')

        # Изменяем первый слой на 1 канал
        in_channels = 1
        self.model._conv_stem = nn.Conv2d(in_channels=in_channels,
                                          out_channels=self.model._conv_stem.out_channels,
                                          kernel_size=self.model._conv_stem.kernel_size,
                                          stride=self.model._conv_stem.stride,
                                          padding=self.model._conv_stem.padding,
                                          bias=False)

        # Выходной слой на 3 класса
        in_features = self.model._fc.in_features
        self.model._fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# Создаём модель
num_classes = 3
model = EfficientNet1C(num_classes).to(device)

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Функция загрузки изображения
def load_mat_image(file_path):
    mat_data = loadmat(file_path)
    if 'DataCubeC' not in mat_data:
        raise KeyError(f"Ключ 'DataCubeC' не найден в файле {file_path}")

    image = mat_data['DataCubeC']  # Извлекаем гиперспектральный куб
    image = np.transpose(image, (2, 0, 1))  # Переставляем оси: (каналы, высота, ширина)
    image = torch.tensor(image, dtype=torch.float32)  # Преобразуем в тензор PyTorch
    return image

# Путь к изображению
image_path = '/content/diplommel1/datamat/train/DNcube/101.mat'
image = load_mat_image(image_path)

# Оставляем только 1 канал (например, первый)
image = image[:1, :, :]  # (1, H, W)

# Добавляем размерность батча
image = image.unsqueeze(0).to(device)  # (1, 1, H, W)

# Определение модели EfficientNet с 16 каналами
class EfficientNet3D(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNet3D, self).__init__()

        self.model = EfficientNet.from_pretrained('efficientnet-b0')

        # Изменяем первый сверточный слой
        in_channels = 16  # Гиперспектральные изображения
        self.model._conv_stem = nn.Conv2d(in_channels=in_channels,
                                          out_channels=self.model._conv_stem.out_channels,
                                          kernel_size=self.model._conv_stem.kernel_size,
                                          stride=self.model._conv_stem.stride,
                                          padding=self.model._conv_stem.padding,
                                          bias=False)

        # Заменяем последний слой для 3 классов
        in_features = self.model._fc.in_features
        self.model._fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# Создаём модель
num_classes = 3
model = EfficientNet3D(num_classes).to(device)
model.eval()

# Выбираем слой для Grad-CAM
target_layer = model.model._conv_head  # Последний сверточный слой

# Инициализируем Grad-CAM
cam_extractor = GradCAM(model, target_layer=model.model._conv_head)

# Пропускаем изображение через модель
output = model(image)
class_idx = output.argmax(dim=1).item()  # Получаем индекс предсказанного класса

# Получаем карту активации
activation_map = cam_extractor(class_idx, output)

# Преобразуем тензоры в numpy
activation_map = activation_map[0].cpu().detach().numpy()
image_np = image[0, 0].cpu().numpy()  # Берём 1 канал для наложения

# **Исправляем ошибку: нормализуем в 0-255 и конвертируем в `PIL.Image`**
image_pil = Image.fromarray((image_np * 255).astype(np.uint8), mode="L")  # "L" = градации серого
activation_pil = Image.fromarray((activation_map * 255).astype(np.uint8), mode="L")

# Накладываем карту активации
overlay = overlay_mask(image_pil.convert("RGB"), activation_pil.convert("RGB"))

# Отображаем результат
plt.imshow(overlay)
plt.axis('off')
plt.show()


Loaded pretrained weights for efficientnet-b0
Loaded pretrained weights for efficientnet-b0


RuntimeError: Given groups=1, weight of size [32, 16, 3, 3], expected input[1, 1, 272, 512] to have 16 channels, but got 1 channels instead